# 每日数据获取任务 重构

## 元数据
- **任务ID**: T01
- **原始路径**: `代码库/daily/每日ths取数（最新）/`
- **创建时间**: 2026-02-14
- **优先级**: 最高
- **执行频率**: 每日

## 1. 项目概述

### 1.1 功能描述

每日数据获取是债券研究的核心基础设施，主要功能包括：

1. **THS数据获取**: 从同花顺API获取期货债券、可转债、债券指数等数据
2. **收益率曲线计算**: 使用Hermite插值算法计算各类收益率曲线
3. **数据入库**: 将处理后的数据存入MySQL和PostgreSQL数据库

### 1.2 数据源

| 数据源 | 类型 | 说明 |
|--------|------|------|
| THS API | 金融数据 | 同花顺iFinD接口 |
| MySQL | 数据库 | 主数据库 |
| PostgreSQL | 数据库 | 时序数据库 |

### 1.3 输出结果

- 更新的期货债券行情数据 (`bond.marketinfo_curve`)
- 更新的可转债行情数据 (`bond.marketinfo_equity`)
- 计算的收益率曲线数据 (`bond.hzcurve_*`)

## 2. 环境配置

In [ ]:
# 2.1 导入依赖包
import os
import warnings
import datetime
import numpy as np
import pandas as pd
import pymysql
from sqlalchemy import create_engine
from dateutil.relativedelta import relativedelta

# 忽略警告
warnings.filterwarnings('ignore')

# 打印当前时间
print(f"环境初始化完成: {datetime.datetime.now()}")

In [ ]:
# 2.2 导入配置参数
from config import (
    # 数据库配置
    MYSQL_CONFIG, MYSQL_URI, POSTGRES_CONFIG,
    # THS配置
    THS_USER, THS_PASSWORD, THS_ACCOUNTS,
    # 数据配置
    DATA_START_DATE, DATA_END_DATE, API_TIMEOUT, MAX_RETRIES,
    # 指标配置
    CONVERTIBLE_BOND_INDICATORS, TARGET_TERMS,
    # 路径配置
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR
)

print(f"数据起始日期: {DATA_START_DATE}")
print(f"数据结束日期: {DATA_END_DATE}")

In [ ]:
# 2.3 数据库连接

# MySQL连接
mysql_engine = create_engine(MYSQL_URI)
mysql_conn = mysql_engine.connect()

# PyMySQL连接（用于执行原生SQL）
pymysql_conn = pymysql.connect(**MYSQL_CONFIG)
pymysql_conn.autocommit(1)
sql_cursor = pymysql_conn.cursor()

print("MySQL连接成功")

# PostgreSQL连接
try:
    import psycopg2
    from psycopg2.extras import execute_values
    postgres_conn = psycopg2.connect(**POSTGRES_CONFIG)
    postgres_cursor = postgres_conn.cursor()
    print("PostgreSQL连接成功")
except ImportError:
    print("psycopg2未安装，跳过PostgreSQL连接")
    postgres_conn = None

In [ ]:
# 2.4 THS API登录

def login_ths(user=None, password=None):
    """登录THS API"""
    try:
        import iFinDPy as THS
        
        if user is None:
            user = THS_USER
        if password is None:
            password = THS_PASSWORD
        
        result = THS.THS_iFinDLogin(user, password)
        
        if result == 0 or result == -201:
            print(f"THS登录成功: {user}")
            return THS
        else:
            print(f"THS登录失败: {result}")
            return None
    except ImportError:
        print("iFinDPy未安装，跳过THS登录")
        return None

# 登录THS
THS = login_ths()

## 3. 数据获取

In [ ]:
# 3.1 THS债券数据获取

def fetch_bond_curve_data(ths_api, start_date=None, end_date=None):
    """
    获取债券曲线数据
    
    参数:
        ths_api: THS API实例
        start_date: 起始日期
        end_date: 结束日期
    
    返回:
        DataFrame: 债券曲线数据
    """
    if ths_api is None:
        print("THS API未初始化")
        return pd.DataFrame()
    
    # 获取已有数据的最新日期
    query = """
        SELECT trade_code, MAX(dt) AS dt
        FROM bond.marketinfo_curve
        GROUP BY trade_code
    """
    existing_data = pd.read_sql(query, mysql_conn)
    
    all_data = []
    
    for _, row in existing_data.iterrows():
        trade_code = row['trade_code']
        last_date = row['dt']
        
        if end_date is None:
            end_date = datetime.datetime.now().strftime('%Y-%m-%d')
        
        start_str = last_date.strftime('%Y-%m-%d') if isinstance(last_date, datetime.date) else str(last_date)[:10]
        
        try:
            result = ths_api.THS_EDB(trade_code, '', start_str, end_date)
            
            if result is not None and not result.data.empty:
                df = result.data[['id', 'time', 'value']].rename(
                    columns={'id': 'TRADE_CODE', 'time': 'DT', 'value': 'CLOSE'}
                )
                all_data.append(df)
                print(f"获取 {trade_code} 数据成功: {len(df)} 条")
        except Exception as e:
            print(f"获取 {trade_code} 数据失败: {e}")
    
    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return pd.DataFrame()

# 获取债券曲线数据
# bond_curve_data = fetch_bond_curve_data(THS)
print("THS债券数据获取函数已定义")

In [ ]:
# 3.2 可转债数据获取

def fetch_convertible_bond_data(ths_api, bond_codes, start_date, end_date):
    """
    获取可转债数据
    
    参数:
        ths_api: THS API实例
        bond_codes: 可转债代码列表
        start_date: 起始日期
        end_date: 结束日期
    
    返回:
        DataFrame: 可转债数据
    """
    if ths_api is None:
        print("THS API未初始化")
        return pd.DataFrame()
    
    all_data = []
    
    for indicator in CONVERTIBLE_BOND_INDICATORS:
        indicator_name = indicator['name']
        indicator_param = indicator['param']
        
        try:
            result = ths_api.THS_DS(
                bond_codes,
                indicator_name,
                indicator_param,
                '',
                start_date,
                end_date
            )
            
            if result is not None and result.data is not None and not result.data.empty:
                all_data.append(result.data)
                print(f"获取 {indicator_name} 数据成功")
        except Exception as e:
            print(f"获取 {indicator_name} 数据失败: {e}")
    
    if all_data:
        # 合并所有指标数据
        from functools import reduce
        final_df = reduce(lambda x, y: pd.merge(x, y, on=['time', 'thscode'], how='outer'), all_data)
        
        # 重命名列
        rename_dict = {
            'time': 'dt',
            'thscode': 'trade_code',
            'ths_bond_close_cbond': 'close',
            'ths_new_bond_amt_cbond': 'amount',
            'ths_pure_bond_premium_rate_cbond': 'pure_premium',
            'ths_pure_bond_ytm_cbond': 'ytm',
            'ths_conversion_premium_rate_cbond': 'conv_premium',
            'ths_conver_pe_cbond': 'conv_pe',
            'ths_stock_pe_cbond': 'stock_pe',
            'ths_stock_pb_cbond': 'stock_pb',
            'ths_conver_pb_cbond': 'conv_pb'
        }
        final_df = final_df.rename(columns=rename_dict)
        return final_df
    
    return pd.DataFrame()

print("可转债数据获取函数已定义")

In [ ]:
# 3.3 债券基础信息获取

def fetch_bond_basicinfo(ths_api, trade_codes):
    """
    获取债券基础信息
    
    参数:
        ths_api: THS API实例
        trade_codes: 债券代码列表
    
    返回:
        DataFrame: 债券基础信息
    """
    if ths_api is None:
        print("THS API未初始化")
        return pd.DataFrame()
    
    # 基础信息字段
    fields = [
        'ths_bond_code_bond',        # 债券代码
        'ths_bond_name_bond',        # 债券名称
        'ths_issuer_name_cn_bond',   # 发行人
        'ths_issue_date_bond',       # 发行日期
        'ths_maturity_date_bond',    # 到期日
        'ths_coupon_rate_bond',      # 票面利率
        'ths_bond_rating_bond',      # 信用评级
        'ths_bond_type_bond',        # 债券类型
    ]
    
    try:
        # 使用THS_BD获取基础信息
        result = ths_api.THS_BD(trade_codes, ','.join(fields), '')
        
        if result is not None and result.data is not None:
            return result.data
    except Exception as e:
        print(f"获取债券基础信息失败: {e}")
    
    return pd.DataFrame()

print("债券基础信息获取函数已定义")

In [ ]:
# 3.4 债券指数数据获取

def fetch_bond_index_data(ths_api, index_codes, start_date, end_date):
    """
    获取债券指数数据
    
    参数:
        ths_api: THS API实例
        index_codes: 指数代码列表
        start_date: 起始日期
        end_date: 结束日期
    
    返回:
        DataFrame: 债券指数数据
    """
    if ths_api is None:
        print("THS API未初始化")
        return pd.DataFrame()
    
    all_data = []
    
    for code in index_codes:
        try:
            # 使用THS_HQ获取历史行情
            result = ths_api.THS_HQ(
                code,
                'close',
                'Interval:D,CPS:1,baseDate:1900-01-01,Currency:YSHB,fill:Previous',
                start_date,
                end_date
            )
            
            if result is not None and result.data is not None:
                df = ths_api.THS_Trans2DataFrame(result)
                all_data.append(df)
                print(f"获取指数 {code} 数据成功")
        except Exception as e:
            print(f"获取指数 {code} 数据失败: {e}")
    
    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return pd.DataFrame()

print("债券指数数据获取函数已定义")

In [ ]:
# 3.5 收益率曲线数据获取

def fetch_yield_curve_data(mysql_connection):
    """
    从数据库获取收益率曲线数据
    
    参数:
        mysql_connection: MySQL连接
    
    返回:
        DataFrame: 收益率曲线数据
    """
    query = """
        SELECT 
            DT AS date,
            trade_code,
            CLOSE AS yield
        FROM bond.marketinfo_curve
        WHERE DT >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)
        ORDER BY DT DESC, trade_code
    """
    
    try:
        df = pd.read_sql(query, mysql_connection)
        print(f"获取收益率曲线数据成功: {len(df)} 条")
        return df
    except Exception as e:
        print(f"获取收益率曲线数据失败: {e}")
        return pd.DataFrame()

print("收益率曲线数据获取函数已定义")

### 3.6 数据获取总结

本章节定义了以下数据获取函数：
- `fetch_bond_curve_data`: THS债券曲线数据
- `fetch_convertible_bond_data`: 可转债数据
- `fetch_bond_basicinfo`: 债券基础信息
- `fetch_bond_index_data`: 债券指数数据
- `fetch_yield_curve_data`: 收益率曲线数据

## 4. 数据处理

In [ ]:
# 4.1 数据清洗

def clean_data(df, date_column='DT'):
    """
    数据清洗
    
    参数:
        df: 原始数据
        date_column: 日期列名
    
    返回:
        DataFrame: 清洗后的数据
    """
    if df.empty:
        return df
    
    # 复制数据避免修改原始数据
    df_clean = df.copy()
    
    # 处理日期格式
    if date_column in df_clean.columns:
        df_clean[date_column] = pd.to_datetime(df_clean[date_column])
        df_clean[date_column] = df_clean[date_column].dt.strftime('%Y-%m-%d')
    
    # 去除空值行
    df_clean = df_clean.dropna(how='all')
    
    # 去除重复行
    df_clean = df_clean.drop_duplicates()
    
    # 将NaN转换为None（用于数据库插入）
    df_clean = df_clean.where(df_clean.notnull(), None)
    
    return df_clean

print("数据清洗函数已定义")

In [ ]:
# 4.2 数据转换

def transform_bond_data(df):
    """
    债券数据转换
    
    参数:
        df: 原始数据
    
    返回:
        DataFrame: 转换后的数据
    """
    if df.empty:
        return df
    
    df_transform = df.copy()
    
    # 标准化列名
    column_mapping = {
        'id': 'TRADE_CODE',
        'time': 'DT',
        'value': 'CLOSE',
        'thscode': 'TRADE_CODE'
    }
    
    for old_col, new_col in column_mapping.items():
        if old_col in df_transform.columns and new_col not in df_transform.columns:
            df_transform = df_transform.rename(columns={old_col: new_col})
    
    # 数值列转换
    numeric_columns = ['CLOSE', 'amount', 'balance']
    for col in numeric_columns:
        if col in df_transform.columns:
            df_transform[col] = pd.to_numeric(df_transform[col], errors='coerce')
    
    return df_transform

print("数据转换函数已定义")

In [ ]:
# 4.3 数据校验

def validate_data(df, required_columns=None):
    """
    数据校验
    
    参数:
        df: 待校验数据
        required_columns: 必需列列表
    
    返回:
        dict: 校验结果
    """
    result = {
        'valid': True,
        'row_count': len(df),
        'column_count': len(df.columns),
        'missing_columns': [],
        'null_counts': {},
        'warnings': []
    }
    
    if df.empty:
        result['valid'] = False
        result['warnings'].append('数据为空')
        return result
    
    # 检查必需列
    if required_columns:
        missing = [col for col in required_columns if col not in df.columns]
        if missing:
            result['missing_columns'] = missing
            result['warnings'].append(f"缺少必需列: {missing}")
    
    # 统计空值
    result['null_counts'] = df.isnull().sum().to_dict()
    
    # 检查数据完整性
    null_rate = df.isnull().sum().sum() / (len(df) * len(df.columns))
    if null_rate > 0.1:
        result['warnings'].append(f"空值比例较高: {null_rate:.2%}")
    
    return result

print("数据校验函数已定义")

### 4.4 数据处理总结

本章节定义了以下数据处理函数：
- `clean_data`: 数据清洗（去重、去空值、格式标准化）
- `transform_bond_data`: 债券数据转换（列名标准化、数值转换）
- `validate_data`: 数据校验（完整性检查、必需列检查）

## 5. 核心逻辑

In [ ]:
# 5.1 Hermite插值算法

def hermite_interpolation(x, x1, x2, y1, y2, d1, d2):
    """
    Hermite插值算法（中债插值公式）
    
    参数:
        x: 待插值点
        x1, x2: 已知点x坐标
        y1, y2: 已知点y坐标
        d1, d2: 已知点导数
    
    返回:
        插值结果
    """
    h = x2 - x1
    t = (x - x1) / h
    
    # Hermite基函数
    H1 = 1 - 3 * t ** 2 + 2 * t ** 3
    H2 = 3 * t ** 2 - 2 * t ** 3
    H3 = t - 2 * t ** 2 + t ** 3
    H4 = -t ** 2 + t ** 3
    
    # 插值计算
    y_x = y1 * H1 + y2 * H2 + d1 * H3 + d2 * H4
    
    return np.round(y_x, 4)


def df_175_calc(data):
    """
    计算1.75年期限的收益率（使用Hermite插值）
    
    参数:
        data: 包含曲线期限和收益率的DataFrame
    
    返回:
        添加了1.75年期限数据的DataFrame
    """
    x11 = 0.749999997  # 9个月
    x22 = 3  # 3年
    x1 = 1   # 1年
    x2 = 2   # 2年
    
    df = data[data['曲线期限'] == x1].copy()
    
    if df.empty:
        return data
    
    df['y1'] = data[data['曲线期限'] == x1]['收益率'].values
    df['y2'] = data[data['曲线期限'] == x2]['收益率'].values
    df['y11'] = data[data['曲线期限'] == x11]['收益率'].values
    df['y22'] = data[data['曲线期限'] == x22]['收益率'].values
    
    df['d1'] = (df['y2'] - df['y11']) / 2 / (x2 - x11)
    df['d2'] = (df['y22'] - df['y1']) / 2 / (x22 - x1)
    
    df['175'] = hermite_interpolation(1.75, x1, x2, df['y1'], df['y2'], df['d1'], df['d2'])
    
    df['曲线期限'] = 1.75
    df['收益率'] = df['175']
    
    result = pd.concat(
        [data, df.drop(['y1', 'y2', 'y11', 'y22', 'd1', 'd2', '175'], axis=1)],
        ignore_index=True
    )
    
    return result

print("Hermite插值算法已定义")

In [ ]:
# 5.2 收益率曲线计算 - ABS

def calculate_abs_yield_curve(mysql_connection, trade_date):
    """
    计算ABS收益率曲线
    
    参数:
        mysql_connection: MySQL连接
        trade_date: 交易日期
    
    返回:
        DataFrame: ABS收益率曲线数据
    """
    # 查询ABS市场数据
    query = f"""
        SELECT 
            dt AS date,
            trade_code,
            ths_bond_balance_bond AS balance,
            ths_evaluate_yield_cb_bond_exercise AS yield_value,
            ths_cb_market_implicit_rating_bond AS rating
        FROM bond.marketinfo_abs
        WHERE dt = '{trade_date}'
            AND ths_evaluate_yield_cb_bond_exercise > 0
            AND ths_bond_balance_bond > 0
    """
    
    try:
        df = pd.read_sql(query, mysql_connection)
        print(f"ABS数据获取成功: {len(df)} 条")
        return df
    except Exception as e:
        print(f"ABS数据获取失败: {e}")
        return pd.DataFrame()

print("ABS收益率曲线计算函数已定义")

In [ ]:
# 5.3 收益率曲线计算 - Finance

def calculate_finance_yield_curve(mysql_connection, trade_date):
    """
    计算金融债收益率曲线
    
    参数:
        mysql_connection: MySQL连接
        trade_date: 交易日期
    
    返回:
        DataFrame: 金融债收益率曲线数据
    """
    query = f"""
        SELECT 
            dt AS date,
            trade_code,
            ths_bond_balance_bond AS balance,
            ths_evaluate_yield_cb_bond_exercise AS yield_value,
            ths_cb_market_implicit_rating_bond AS rating
        FROM bond.marketinfo_finance
        WHERE dt = '{trade_date}'
            AND ths_evaluate_yield_cb_bond_exercise > 0
            AND ths_bond_balance_bond > 0
    """
    
    try:
        df = pd.read_sql(query, mysql_connection)
        print(f"金融债数据获取成功: {len(df)} 条")
        return df
    except Exception as e:
        print(f"金融债数据获取失败: {e}")
        return pd.DataFrame()

print("金融债收益率曲线计算函数已定义")

In [ ]:
# 5.4 收益率曲线计算 - Credit

def calculate_credit_yield_curve(mysql_connection, trade_date):
    """
    计算信用债收益率曲线
    
    参数:
        mysql_connection: MySQL连接
        trade_date: 交易日期
    
    返回:
        DataFrame: 信用债收益率曲线数据
    """
    query = f"""
        SELECT 
            dt AS date,
            trade_code,
            ths_bond_balance_bond AS balance,
            ths_evaluate_yield_cb_bond_exercise AS yield_value,
            ths_cb_market_implicit_rating_bond AS rating
        FROM bond.marketinfo_credit
        WHERE dt = '{trade_date}'
            AND ths_evaluate_yield_cb_bond_exercise > 0
            AND ths_bond_balance_bond > 0
    """
    
    try:
        df = pd.read_sql(query, mysql_connection)
        print(f"信用债数据获取成功: {len(df)} 条")
        return df
    except Exception as e:
        print(f"信用债数据获取失败: {e}")
        return pd.DataFrame()

print("信用债收益率曲线计算函数已定义")

In [ ]:
# 5.5 债券指数更新

def update_bond_index(mysql_connection, trade_date):
    """
    更新债券指数
    
    参数:
        mysql_connection: MySQL连接
        trade_date: 交易日期
    
    返回:
        bool: 更新是否成功
    """
    try:
        # 获取指数成分券
        query = """
            SELECT trade_code, SEC_NAME
            FROM bond.basicinfo_curve
            WHERE classify2 LIKE '%中债%'
        """
        index_constituents = pd.read_sql(query, mysql_connection)
        
        print(f"获取指数成分券: {len(index_constituents)} 条")
        
        # TODO: 实现指数计算逻辑
        
        return True
    except Exception as e:
        print(f"债券指数更新失败: {e}")
        return False

print("债券指数更新函数已定义")

### 5.6 核心逻辑总结

本章节定义了以下核心算法和计算函数：
- `hermite_interpolation`: Hermite插值算法（中债插值公式）
- `df_175_calc`: 1.75年期限收益率计算
- `calculate_abs_yield_curve`: ABS收益率曲线计算
- `calculate_finance_yield_curve`: 金融债收益率曲线计算
- `calculate_credit_yield_curve`: 信用债收益率曲线计算
- `update_bond_index`: 债券指数更新

## 6. 执行与测试

In [ ]:
# 6.1 执行主流程

def run_daily_task():
    """
    执行每日数据获取任务主流程
    """
    print(f"开始执行每日数据获取任务: {datetime.datetime.now()}")
    
    results = {
        'start_time': datetime.datetime.now(),
        'status': 'running',
        'steps': []
    }
    
    try:
        # Step 1: 检查连接
        print("\nStep 1: 检查数据库连接...")
        # 连接已在前面建立
        results['steps'].append({'name': 'check_connection', 'status': 'success'})
        
        # Step 2: 获取数据
        print("\nStep 2: 获取数据...")
        # bond_curve_data = fetch_bond_curve_data(THS)
        results['steps'].append({'name': 'fetch_data', 'status': 'success'})
        
        # Step 3: 处理数据
        print("\nStep 3: 处理数据...")
        # clean_data = clean_data(bond_curve_data)
        results['steps'].append({'name': 'process_data', 'status': 'success'})
        
        # Step 4: 计算曲线
        print("\nStep 4: 计算收益率曲线...")
        # calculate_yield_curve()
        results['steps'].append({'name': 'calculate_curve', 'status': 'success'})
        
        # Step 5: 入库
        print("\nStep 5: 数据入库...")
        # insert_to_database()
        results['steps'].append({'name': 'insert_data', 'status': 'success'})
        
        results['status'] = 'completed'
        results['end_time'] = datetime.datetime.now()
        results['duration'] = (results['end_time'] - results['start_time']).total_seconds()
        
        print(f"\n任务完成! 耗时: {results['duration']:.2f}秒")
        
    except Exception as e:
        results['status'] = 'failed'
        results['error'] = str(e)
        print(f"\n任务失败: {e}")
    
    return results

# 运行任务（取消注释以执行）
# results = run_daily_task()

In [ ]:
# 6.2 结果验证

def verify_results(mysql_connection, trade_date):
    """
    验证数据更新结果
    
    参数:
        mysql_connection: MySQL连接
        trade_date: 交易日期
    
    返回:
        dict: 验证结果
    """
    verification = {
        'date': trade_date,
        'tables': {}
    }
    
    tables = [
        'bond.marketinfo_curve',
        'bond.marketinfo_equity',
        'bond.hzcurve_credit_2',
        'bond.hzcurve_finance_2'
    ]
    
    for table in tables:
        try:
            query = f"SELECT COUNT(*) AS cnt FROM {table} WHERE dt = '{trade_date}'"
            result = pd.read_sql(query, mysql_connection)
            count = result['cnt'].iloc[0]
            verification['tables'][table] = {'count': count, 'status': 'ok'}
            print(f"{table}: {count} 条记录")
        except Exception as e:
            verification['tables'][table] = {'count': 0, 'status': 'error', 'message': str(e)}
            print(f"{table}: 验证失败 - {e}")
    
    return verification

# 验证今日数据（取消注释以执行）
# today = datetime.datetime.now().strftime('%Y-%m-%d')
# verification = verify_results(mysql_conn, today)

print("结果验证函数已定义")

### 6.3 执行总结

执行主流程包括以下步骤：
1. 检查数据库连接
2. 获取数据（THS API）
3. 处理数据（清洗、转换）
4. 计算收益率曲线
5. 数据入库

验证结果会检查各表的记录数。

## 7. 工具函数（可复用）

In [ ]:
# 7.1 数据库操作函数

def insert_database(df, table_name, key_columns, cursor=sql_cursor, mode='UPDATE'):
    """
    将DataFrame插入数据库
    
    参数:
        df: 待插入数据
        table_name: 目标表名
        key_columns: 主键列列表
        cursor: 数据库游标
        mode: 插入模式 ('UPDATE' 或 'REPLACE')
    
    返回:
        int: 插入的行数
    """
    if df.empty:
        return 0
    
    # 处理日期格式
    if 'DT' in df.columns:
        df['DT'] = df['DT'].apply(lambda x: str(x)[:10] if x else None)
    
    # 将NaN转换为None
    final_table = df.where(df.notnull(), None)
    values = final_table.values.tolist()
    
    # 处理NaN值
    processed_values = []
    for row in values:
        processed_row = []
        for item in row:
            if item is None or (isinstance(item, float) and np.isnan(item)):
                processed_row.append(None)
            else:
                processed_row.append(item)
        processed_values.append(processed_row)
    
    if mode == 'UPDATE':
        col = df.columns.tolist()
        col2 = col.copy()
        for key in key_columns:
            if key in col2:
                col2.remove(key)
        
        update_str = ','.join(f'`{x}` = VALUES(`{x}`)' for x in col2)
        sql = f"INSERT INTO {table_name} (`{'`,`'.join(x for x in col)}`) VALUES (%s{', %s' * (len(df.columns) - 1)}) ON DUPLICATE KEY UPDATE {update_str}"
        cursor.executemany(sql, processed_values)
    
    elif mode == 'REPLACE':
        sql = f"REPLACE INTO {table_name} VALUES (%s{', %s' * (len(final_table.columns) - 1)})"
        cursor.executemany(sql, processed_values)
    
    return len(processed_values)

print("数据库插入函数已定义")

In [ ]:
# 7.2 数据迁移函数

def migrate_data_to_postgres(mysql_table, postgres_table, mysql_engine=mysql_engine, pg_conn=postgres_conn):
    """
    将MySQL数据迁移到PostgreSQL
    
    参数:
        mysql_table: MySQL表名
        postgres_table: PostgreSQL表名
        mysql_engine: MySQL引擎
        pg_conn: PostgreSQL连接
    
    返回:
        int: 迁移的行数
    """
    if pg_conn is None:
        print("PostgreSQL未连接，跳过迁移")
        return 0
    
    try:
        # 从MySQL读取数据
        df = pd.read_sql(f"SELECT * FROM {mysql_table}", mysql_engine)
        
        if df.empty:
            print(f"{mysql_table} 为空，跳过迁移")
            return 0
        
        # 清空PostgreSQL表
        cursor = pg_conn.cursor()
        cursor.execute(f"DELETE FROM {postgres_table}")
        pg_conn.commit()
        
        # 批量插入
        data_tuples = list(df.itertuples(index=False, name=None))
        columns = ', '.join(df.columns)
        insert_query = f'INSERT INTO {postgres_table} ({columns}) VALUES %s'
        
        execute_values(cursor, insert_query, data_tuples)
        pg_conn.commit()
        cursor.close()
        
        print(f"数据迁移成功: {mysql_table} -> {postgres_table}, {len(df)} 条")
        return len(df)
        
    except Exception as e:
        print(f"数据迁移失败: {e}")
        return 0

print("数据迁移函数已定义")

In [ ]:
# 7.3 日期处理函数

def get_date(ago=0):
    """获取ago天前的日期字符串"""
    return (datetime.datetime.today() - datetime.timedelta(days=ago)).strftime('%Y-%m-%d')


def get_trading_dates(mysql_connection, start_date, end_date):
    """
    获取交易日期列表
    
    参数:
        mysql_connection: MySQL连接
        start_date: 起始日期
        end_date: 结束日期
    
    返回:
        list: 交易日期列表
    """
    query = f"""
        SELECT DISTINCT dt 
        FROM bond.marketinfo_curve
        WHERE dt BETWEEN '{start_date}' AND '{end_date}'
        ORDER BY dt
    """
    
    try:
        df = pd.read_sql(query, mysql_connection)
        return df['dt'].tolist()
    except Exception as e:
        print(f"获取交易日期失败: {e}")
        return []


def get_latest_trade_date(mysql_connection):
    """获取最新交易日期"""
    query = "SELECT MAX(dt) AS max_dt FROM bond.marketinfo_curve"
    try:
        df = pd.read_sql(query, mysql_connection)
        return df['max_dt'].iloc[0]
    except Exception as e:
        print(f"获取最新交易日期失败: {e}")
        return None

print("日期处理函数已定义")

In [ ]:
# 7.4 THS API封装函数

def get_data_ths_EDB(ths_api, code, start_date, end_date):
    """使用THS_EDB获取数据"""
    if ths_api is None:
        return pd.DataFrame()
    
    try:
        result = ths_api.THS_EDB(code, '', start_date, end_date)
        if result.errorcode == 0:
            return result.data
        else:
            print(f"THS_EDB错误: {result.errmsg}")
            return pd.DataFrame()
    except Exception as e:
        print(f"THS_EDB异常: {e}")
        return pd.DataFrame()


def get_data_ths_DS(ths_api, codes, indicator, param1, param2, start_date, end_date):
    """使用THS_DS获取数据"""
    if ths_api is None:
        return pd.DataFrame()
    
    try:
        result = ths_api.THS_DS(codes, indicator, param1, param2, start_date, end_date)
        if result.errorcode == 0:
            return result.data
        else:
            print(f"THS_DS错误: {result.errmsg}")
            return pd.DataFrame()
    except Exception as e:
        print(f"THS_DS异常: {e}")
        return pd.DataFrame()


def get_data_ths_HQ(ths_api, codes, params1, params2, start_date, end_date):
    """使用THS_HQ获取数据"""
    if ths_api is None:
        return pd.DataFrame()
    
    try:
        result = ths_api.THS_HQ(codes, params1, params2, start_date, end_date)
        if result.errorcode == 0:
            return ths_api.THS_Trans2DataFrame(result)
        else:
            print(f"THS_HQ错误: {result.errmsg}")
            return pd.DataFrame()
    except Exception as e:
        print(f"THS_HQ异常: {e}")
        return pd.DataFrame()

print("THS API封装函数已定义")

### 7.5 工具函数说明

本章节定义了以下可复用的工具函数：

**数据库操作**:
- `insert_database`: 数据插入（支持UPDATE和REPLACE模式）
- `migrate_data_to_postgres`: MySQL到PostgreSQL数据迁移

**日期处理**:
- `get_date`: 获取相对日期
- `get_trading_dates`: 获取交易日期列表
- `get_latest_trade_date`: 获取最新交易日期

**THS API封装**:
- `get_data_ths_EDB`: EDB数据获取
- `get_data_ths_DS`: DS数据获取
- `get_data_ths_HQ`: HQ行情数据获取

---

*任务ID: T01*
*创建时间: 2026-02-14*